# Implementación inicial del proyecto: nanotoxicidad

Esta notebook arranca la implementación funcional del proyecto integrador de la Unidad 6.

Objetivo:
- preparar un flujo reproducible para toxicidad de nanopartículas
- dejar lista una versión inicial compatible con U6_03_IMPLEMENTACION_PROYECTO.ipynb
- conectar el dataset del curso con una estructura que luego pueda integrar eNanoMapper u otra base pública de toxicidad
- construir un baseline binario simple: toxic / non-toxic

Importante:
- la etiqueta binaria generada aquí es provisional y sirve para arrancar el pipeline
- cuando tengas una base pública con labels reales de toxicidad, debes reemplazar esta etapa de etiquetado por la fuente experimental real

## 1. Estrategia de trabajo

La implementación inicial se divide en 5 bloques:

1. Cargar el dataset disponible en el repositorio.
2. Inspeccionar variables numéricas, categóricas y valores faltantes.
3. Construir una etiqueta binaria provisional para bootstrapping.
4. Preparar el preprocesamiento y el modelo baseline.
5. Guardar un dataset limpio y una versión lista para U6_03.

Esta estructura mantiene compatibilidad con la Unidad 6 y con la futura integración multiagente.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

sns.set_theme(style='whitegrid')

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
FIGURES_DIR = ROOT / 'figuras'
for folder in [DATA_DIR, RAW_DIR, PROCESSED_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('Carpetas listas:')
print('-', RAW_DIR)
print('-', PROCESSED_DIR)
print('-', FIGURES_DIR)

## 2. Carga del dataset

Primero se intenta cargar una base pública de toxicidad si ya fue colocada en la carpeta del proyecto.

Si no existe todavía, se usa el dataset estructural del curso como respaldo para avanzar con el pipeline.

Ruta esperada para una base pública futura:
- `data/raw/eNanoMapper/`
- `data/raw/nanotoxicity/`

Dataset de respaldo actual:
- `educational_content/unit_03_ml_nanomaterials/nanomaterials_full_dataset.csv`

In [ ]:
def load_first_existing(paths):
    for path in paths:
        if path.exists():
            return path
    return None

candidate_paths = [
    RAW_DIR / 'eNanoMapper' / 'nanotoxicity.csv',
    RAW_DIR / 'eNanoMapper' / 'enanomapper.csv',
    RAW_DIR / 'nanotoxicity.csv',
    Path.cwd().parent / 'unit_03_ml_nanomaterials' / 'nanomaterials_full_dataset.csv',
    Path('c:/Users/natal/OneDrive/Documentos/PROYECTO IA/Antigravity-Nano-Research-Multiagentic-Core/educational_content/unit_03_ml_nanomaterials/nanomaterials_full_dataset.csv'),
]

dataset_path = load_first_existing(candidate_paths)
if dataset_path is None:
    raise FileNotFoundError('No se encontró ningún dataset candidato en las rutas esperadas.')

df = pd.read_csv(dataset_path)
print(f'Dataset cargado: {dataset_path}')
print(f'Forma: {df.shape[0]} filas x {df.shape[1]} columnas')
df.head()

## 3. Inspección inicial

En esta etapa identificamos:
- variables numéricas
- variables categóricas
- valores faltantes
- duplicados

Esto será reutilizable tanto en U6_03 como en la integración multiagente.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print('Columnas numéricas:')
print(numeric_cols)
print('\nColumnas categóricas:')
print(categorical_cols)
print('\nValores faltantes por columna:')
print(df.isna().sum().sort_values(ascending=False))
print('\nDuplicados:', df.duplicated().sum())

display(df[numeric_cols].describe().T if numeric_cols else pd.DataFrame())

for col in categorical_cols:
    print(f'\nFrecuencias de {col}:')
    print(df[col].value_counts(dropna=False).head(10))

## 4. Preparación de una etiqueta binaria provisional

Para comenzar con una implementación funcional, se define una etiqueta provisional `target_binary`.

Esta etiqueta no reemplaza una anotación experimental real de toxicidad. Solo sirve para arrancar el flujo de entrenamiento, validación y API mientras se integra una base pública mejor etiquetada.

Criterio provisional:
- elementos o grupos asociados con mayor riesgo se marcan como `toxic`
- el resto se marcan como `non_toxic`

Cuando llegue el dataset definitivo de toxicidad, esta columna debe ser sustituida por la etiqueta real.

In [ ]:
def assign_provisional_label(row):
    toxic_elements = {'Hg', 'Pb', 'Cd', 'As', 'Cr', 'Ni', 'Co', 'Cu', 'Zn'}
    toxic_groups = {'heavy_metal', 'transition_metal'}

    element = str(row.get('element', '')).strip()
    element_group = str(row.get('element_group', '')).strip().lower()

    if element in toxic_elements or element_group in toxic_groups:
        return 'toxic'
    return 'non_toxic'

if 'target_binary' not in df.columns:
    df['target_binary'] = df.apply(assign_provisional_label, axis=1)

print(df['target_binary'].value_counts(dropna=False))
df[['element', 'element_group', 'target_binary']].head(10)

## 5. Limpieza mínima y dataset utilizable

Para una primera versión simple:
- eliminamos filas duplicadas
- normalizamos nombres de columnas
- filtramos columnas vacías si existieran
- guardamos una copia limpia en `data/processed/`

El objetivo aquí no es hacer limpieza exhaustiva, sino dejar un dataset consistente para entrenar un baseline.

In [ ]:
df_clean = df.copy()
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
df_clean.columns = [c.strip().lower() for c in df_clean.columns]

empty_cols = [c for c in df_clean.columns if df_clean[c].isna().all()]
if empty_cols:
    df_clean = df_clean.drop(columns=empty_cols)

clean_path = PROCESSED_DIR / 'nanotoxicity_ready.csv'
df_clean.to_csv(clean_path, index=False)

print(f'Dataset limpio guardado en: {clean_path}')
print(f'Forma limpia: {df_clean.shape[0]} filas x {df_clean.shape[1]} columnas')
print('Valores faltantes por columna:')
print(df_clean.isna().sum().sort_values(ascending=False).head(20))

## 6. Exploración visual básica

Estas gráficas ayudan a documentar la estructura del dataset y sirven como punto de partida para el reporte final.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.ravel()

if 'n_atoms' in df_clean.columns:
    sns.histplot(df_clean['n_atoms'], kde=True, ax=axes[0], color='steelblue')
    axes[0].set_title('Distribución de n_atoms')

if 'energy_per_atom' in df_clean.columns:
    sns.histplot(df_clean['energy_per_atom'], kde=True, ax=axes[1], color='darkorange')
    axes[1].set_title('Distribución de energy_per_atom')

if 'geometry' in df_clean.columns:
    order = df_clean['geometry'].value_counts().index
    sns.countplot(data=df_clean, y='geometry', order=order, ax=axes[2], color='slateblue')
    axes[2].set_title('Frecuencia por geometry')

sns.countplot(data=df_clean, x='target_binary', ax=axes[3], palette='Set2')
axes[3].set_title('Distribución del target provisional')

plt.tight_layout()
fig_path = FIGURES_DIR / 'eda_basico.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada en: {fig_path}')

## 7. Preparación del pipeline para U6_03

Ahora se separan features y target para dejar la base lista para entrenamiento.

Este bloque es compatible con la estructura de la Unidad 6 y con un posterior despliegue en FastAPI.

In [ ]:
target_col = 'target_binary'
feature_cols = [c for c in df_clean.columns if c != target_col]
X = df_clean[feature_cols].copy()
y = df_clean[target_col].copy()

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y if y.nunique() > 1 else None
)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

print('Features numéricas:', numeric_features)
print('Features categóricas:', categorical_features)
print('Tamaño train:', X_train.shape, y_train.shape)
print('Tamaño test:', X_test.shape, y_test.shape)

## 8. Baseline de clasificación binaria

Se usa regresión logística como primer modelo por simplicidad, interpretabilidad y compatibilidad con U6.

Cuando se integre una base más robusta, este baseline puede compararse con Random Forest o XGBoost.

In [ ]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

if hasattr(model.named_steps['classifier'], 'predict_proba'):
    y_proba = model.predict_proba(X_test)[:, 1]
else:
    y_proba = None

metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred, pos_label='toxic', zero_division=0),
    'recall': recall_score(y_test, y_pred, pos_label='toxic', zero_division=0),
    'f1': f1_score(y_test, y_pred, pos_label='toxic', zero_division=0),
}
if y_proba is not None and y_test.nunique() > 1:
    try:
        metrics['roc_auc'] = roc_auc_score((y_test == 'toxic').astype(int), y_proba)
    except Exception:
        metrics['roc_auc'] = np.nan

print('Métricas del baseline:')
for k, v in metrics.items():
    print(f'- {k}: {v:.4f}')

print('\nReporte de clasificación:')
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=['non_toxic', 'toxic'])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['non_toxic', 'toxic'], yticklabels=['non_toxic', 'toxic'])
plt.title('Matriz de confusión del baseline')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Conexión con `toxicity_predictor.py` y el sistema multiagente

`external_skills.ai_mining.toxicity_predictor` se puede usar como una capa de validación rápida o safety gate.

En esta fase inicial, su papel es:
- ofrecer una heurística alternativa
- marcar candidatos de alto riesgo
- ayudar a probar el flujo multiagente antes de tener el dataset final completamente integrado

Luego, el sistema puede organizarse así:
- Orchestrator: decide qué tarea corre primero
- Data Agent: carga y limpia datos
- Model Agent: entrena y evalúa
- Safety Gate: consulta `toxicity_predictor.py`
- Report Agent: resume resultados y genera entregables

In [ ]:
summary = {
    'dataset': str(dataset_path),
    'shape': df_clean.shape,
    'numeric_cols': numeric_cols,
    'categorical_cols': categorical_cols,
    'target_counts': df_clean['target_binary'].value_counts().to_dict(),
    'metrics': {k: float(v) if pd.notnull(v) else None for k, v in metrics.items()},
}

print(summary)

summary_path = PROCESSED_DIR / 'nanotoxicity_summary.json'
import json
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Resumen guardado en: {summary_path}')

## 10. Próximo paso en Unidad 6

Con esta notebook ya tienes un flujo funcional inicial.

El siguiente paso es llevar este pipeline a `U6_03_IMPLEMENTACION_PROYECTO.ipynb` para:
- sustituir la etiqueta provisional por una etiqueta real de toxicidad
- probar un modelo más fuerte
- conectar la salida con la API FastAPI de `mi_proyecto_api/`
- integrar el Safety Gate con `toxicity_predictor.py` dentro del sistema multiagente